# unit05 レッスン: 堅牢化とCSV出力

**このレッスンで作れるようになるもの**: 実サイトから拾った「汚れたデータ」(全角空白・`￥1,200`のような通貨表記・欠損セル)を整った値に正規化し、壊れたレコードは安全にスキップし、一時的な通信失敗にはリトライで粘り、最後に `list[dict]` を **Excelで開けるUTF-8のCSVファイル** に書き出す — スクレイパーの「仕上げ工程」一式。

> スクレイパーは、書いた日の夜に壊れます。テスト用に選んだきれいなページでは動いても、実データは必ず汚れています(前後にベッタリ付いた空白、`¥`やカンマ付きの価格、ページごとにバラバラな日付表記、そもそも空っぽのセル)。さらに通信は時々タイムアウトします。**この汚れと失敗を前提に組むのが「堅牢化」** です。地味ですが、実務のスクレイピングの8割はこの仕上げ作業です。

- 所要時間: 15〜25分
- 進め方: セルを上から順に実行(`Shift+Enter`)。「書いてみる」セルだけ自分で書く
- 詰まったら: Claude に聞いてOK(答えではなくヒントをくれます)

In [ ]:
def check(name, actual, expected, hint=""):
    import numpy as _np
    try:
        ok = actual is not None and bool(_np.all(_np.isclose(_np.asarray(actual, dtype=float), _np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok

import re, csv, io, tempfile, os

# このレッスンで題材にする「汚れた入荷記録」HTML(演習の data/dirty_records.html と同じ内容)。
# 本物なら requests で取得し BeautifulSoup で解析した後の値ですが、ここでは
# 「解析済みだが、まだ汚れている状態」を手元の文字列として用意しておきます。
# 注目ポイント: 前後の空白 / 全角スペース(　)/ ￥・¥・円・カンマ / 空っぽの <span> が混在。
DIRTY_HTML = """<ul class="record-list">
    <li class="record">
        <span class="name">  エチオピア  イルガチェフェ </span>
        <span class="price">￥1,200</span>
        <span class="date">2024/1/5</span>
        <span class="stock">12</span>
    </li>
    <li class="record">
        <span class="name">グアテマラ　アンティグア</span>
        <span class="price"> 980円 </span>
        <span class="date">2024-01-10</span>
        <span class="stock"></span>
    </li>
    <li class="record">
        <span class="name">ブラジル セラード</span>
        <span class="price">¥ 1,500</span>
        <span class="date">2024年1月15日</span>
        <span class="stock">7</span>
    </li>
</ul>"""

print("準備OK! DIRTY_HTML は", len(DIRTY_HTML), "文字")

---
## 概念1: データクレンジング — 「ゴミを入れればゴミが出る」

### なぜ学ぶか
解析([2])で切り出したばかりの値は、たいてい汚れています。`"  エチオピア  イルガチェフェ "` のように前後や内部に空白が残り、価格は `"￥1,200"` のように**人間向けの装飾**が付き、日付は `"2024/1/5"` `"2024年1月15日"` とページごとに表記がバラバラ。これをそのままCSVに書くと、Excelで開いたとき価格が**文字列扱い**になって合計が計算できなかったり、日付でソートできなかったりします。「Garbage In, Garbage Out(ゴミを入れればゴミが出る)」— 出力の品質は、この整形工程で決まります。

### 解説

クレンジングは、C# で外部APIのレスポンスをDTOに詰める前にかける**バリデーション/正規化**と同じ工程です。今日使う道具は文字列メソッドと正規表現の3点:

| Python | 何をする | C# の対応 |
|--------|----------|-----------|
| `s.strip(chars)` | 前後の指定文字を除去(引数無しなら空白類) | `s.Trim(chars)` |
| `s.replace(a, b)` | `a` を `b` に置換(全部) | `s.Replace(a, b)` |
| `re.sub(pat, repl, s)` | 正規表現 `pat` に当たる箇所を `repl` に置換 | `Regex.Replace` |

**新顔は `re.sub`(正規表現置換)**です。`re` は Python標準の正規表現モジュール(C# の `System.Text.RegularExpressions`)。`re.sub(r"[ 　]+", " ", s)` は「**半角スペースか全角スペース(`　`)が1個以上連続**する箇所を、半角スペース1個に畳み込む」という意味です。

- `r"..."` は**raw文字列**。`\` をエスケープせずそのまま書ける記法で、正規表現では定番(C# の逐語的文字列 `@"..."` に相当)。
- `[ 　]` は「この中のどれか1文字」(角括弧の中に半角スペースと全角スペースを並べている)。`+` は「直前を1個以上」。

**通貨→数値**の定石は、装飾を全部 `replace` で消してから `int()`(文字列→整数変換、C# の `int.Parse`):
```python
int("￥1,200".replace("￥", "").replace(",", ""))   # → 1200
```

**日付の正規化**は表記揺れ(`/` `-` `年月日`)を吸収する必要があるので `re.match`(先頭からのパターン照合)を使います。詳しくは worked example で。

In [ ]:
# GOAL: 汚れた3種類の値(名前・価格・日付)を、きれいな形に正規化する手触りを掴む

# STEP 1: 空白の正規化 — strip で前後を落とし、re.sub で内部の連続空白を1つに畳む
raw_name = "  エチオピア  イルガチェフェ "
stripped = raw_name.strip(" 　")            # 前後の半角/全角スペースを除去
name = re.sub(r"[ 　]+", " ", stripped)      # 内部の連続空白 → 半角1つ
print("名前 :", repr(raw_name), "->", repr(name))

# STEP 2: 通貨表記 → 整数。装飾(￥ ¥ 円 , と空白)を replace で消してから int()
raw_price = "￥1,200"
cleaned = raw_price.replace("￥", "").replace("¥", "").replace("円", "").replace(",", "").strip()
price = int(cleaned)                          # 文字列 "1200" -> 整数 1200
print("価格 :", repr(raw_price), "->", price, "(型:", type(price).__name__, ")")

# STEP 3: 日付の表記揺れを YYYY-MM-DD に。re.match で「年 区切り 月 区切り 日」を取り出す
#         [/\-年] は「/ か - か 年」のどれか1文字。(\d{1,2}) は数字1〜2桁の取り出し。
raw_date = "2024年1月15日"
m = re.match(r"(\d{4})[/\-年](\d{1,2})[/\-月](\d{1,2})日?", raw_date)
y, mo, d = m.groups()                          # ("2024", "1", "15") が取れる
date = f"{int(y):04d}-{int(mo):02d}-{int(d):02d}"   # ゼロ埋めして連結
print("日付 :", repr(raw_date), "->", date)

### 予測してみよう

次のセルは、**全角スペースを含む** `"グアテマラ　アンティグア"`(区切りが全角スペース `　`)を STEP 1 と同じ手順で正規化します。

**実行する前に予測**: `re.sub(r"[ 　]+", " ", ...)` の `[ 　]` には全角スペースも入っています。結果は `"グアテマラ　アンティグア"` のままでしょうか、それとも半角スペース区切りの `"グアテマラ アンティグア"` になるでしょうか?

In [ ]:
# 予測してから実行!
raw = "グアテマラ　アンティグア"      # 区切りは全角スペース
result = re.sub(r"[ 　]+", " ", raw.strip(" 　"))
print("正規化前:", repr(raw))
print("正規化後:", repr(result))

全角スペースも `[ 　]` に含めておいたので、ちゃんと半角スペースに揃いました。実データは半角/全角が混ざるので、両方を対象にするのが定石です。

### 書いてみる

**課題**: 汚れた価格文字列 `" 980円 "`(前後に空白、末尾に「円」)を整数に変換して `result1` に入れてください(期待値: `980`)。

ヒント(概念レベル): worked example の STEP 2 と同じ。`"円"` と余分な空白を消してから `int(...)` で変換。今回は `￥` や `,` は含まれませんが、消そうとしても害はありません。

In [ ]:
raw_price = " 980円 "

result1 = None
# ここに書く(result1 に代入する。装飾と空白を消してから int で数値化)


check("概念1: 通貨表記を数値に", result1, 980,
      hint='raw_price.replace("円", "").strip() で "980" にしてから int(...) で囲む')

---
## 概念2: 例外処理とリトライ — 欠損に耐え、一時的な失敗に粘る

### なぜ学ぶか
実データには**空っぽのセル**(`<span class="stock"></span>`)や**丸ごと欠けた要素**が必ず混じります。これを普通に `int("")` しようとすると例外で**スクレイパー全体が停止**します。1件の汚れで全体が落ちるのは論外なので、「変換できないなら既定値でしのぐ」防御を仕込みます。加えて、ネットワーク越しの取得は**一時的に失敗**します(タイムアウト、瞬間的な503)。少し待って**リトライ**すれば成功することが多いので、そのための粘りも用意します。

### 解説

#### (a) 例外を投げない変換 — `safe_int`
C# には `int.TryParse(s, out var v)` という「例外を投げない版」がありますが、Python の `int()` は失敗すると `ValueError` を投げます。そこで `try/except`(C# の `try/catch`)で包んで**フォールバック値**を返す関数を自分で作ります:

```python
def safe_int(text, default=0):
    try:
        return int(text)
    except (TypeError, ValueError):   # 変換失敗も None も、まとめて拾う
        return default
```
`int("")` は `ValueError`、`int(None)` は `TypeError`。この2つを `except` で受けて `default` を返せば、空セルでも落ちません。

#### (b) リトライ(指数バックオフ + ジッタ)
一時的な失敗に対しては、**待ち時間を試行ごとに倍々に伸ばしながら**再試行します。これを**指数バックオフ**と呼びます(C# の `Polly` ライブラリの `WaitAndRetry` と同じ発想):

```
1回目失敗 → 0.1秒待つ → 2回目失敗 → 0.2秒待つ → 3回目失敗 → 0.4秒待つ → ...
待ち時間 = base_delay * (2 ** 試行回数)
```
`2 ** n` は「2のn乗」(C# の `Math.Pow(2, n)`)。倍々にするのは、相手サーバが混んでいるとき**畳みかけずに間隔を空けて**回復を待つためです。さらに実務では**ジッタ**(小さな乱数の揺らぎ)を足します。全クライアントがピッタリ同じ間隔で再試行すると再び集中して混雑するので、少しずらすためです。

> **設計のキモ: 待機処理を「注入」する。** `time.sleep` を関数内に直接書くと、テストのたびに本当に数秒待たされます。そこで「待つ関数 `sleep_fn` を引数で受け取る」形にしておくと、テストでは**何もしないダミー関数**を渡して一瞬で検証できます。C# でいう依存性注入(DI)で `Task.Delay` をモックに差し替えるのと同じ発想です。

In [ ]:
# GOAL: 空セルでも落ちない safe_int と、指数バックオフで粘るリトライの動きを見る

# STEP 1: safe_int — 変換できない入力でも例外で止まらず default が返る
def safe_int(text, default=0):
    try:
        return int(text)
    except (TypeError, ValueError):
        return default

print("safe_int('7')  ->", safe_int("7"))       # 正常に 7
print("safe_int('')   ->", safe_int(""))         # 空文字 -> default 0
print("safe_int(None) ->", safe_int(None))       # None -> default 0

# STEP 2: リトライ本体。sleep_fn を注入できる形にして、待ち時間を記録して観察する
def retry_with_backoff(func, max_retries=3, base_delay=0.1, sleep_fn=None):
    for attempt in range(max_retries):
        try:
            return func()                        # 成功したら即 return
        except Exception:
            if attempt == max_retries - 1:       # 最後の試行なら諦めて再送出
                raise
            delay = base_delay * (2 ** attempt)  # 指数バックオフ: 0.1, 0.2, 0.4, ...
            sleep_fn(delay)                       # 実際の待機は注入された関数に任せる

# STEP 3: 「2回失敗して3回目に成功する」関数で試す。sleep_fn は待ち時間を記録するだけのダミー
waits = []
calls = {"n": 0}
def flaky():
    calls["n"] += 1
    if calls["n"] < 3:
        raise TimeoutError("一時的な失敗")
    return "取得成功!"

result = retry_with_backoff(flaky, sleep_fn=lambda d: waits.append(d))
print("結果   :", result, "/ 呼び出し回数:", calls["n"])
print("待ち時間:", waits, "(← 倍々に伸びている)")

### 予測してみよう

次のセルは `retry_with_backoff` を、**毎回必ず失敗する**関数に対して `max_retries=3` で呼びます。`sleep_fn` はダミー(実際には待ちません)。

**実行する前に予測**: (1) 関数は何回呼ばれるでしょう? (2) 3回とも失敗したとき、最後はどうなるでしょう(例外が出る? それとも None が返る?)。worked example の「最後の試行なら `raise`」に注目して考えてください。

In [ ]:
# 予測してから実行!
count = {"n": 0}
def always_fail():
    count["n"] += 1
    raise ConnectionError("ずっと失敗")

try:
    retry_with_backoff(always_fail, max_retries=3, sleep_fn=lambda d: None)
except ConnectionError as e:
    print("最終的に例外が再送出された:", e)
print("関数が呼ばれた回数:", count["n"])

3回試して全部ダメなら、最後の例外をそのまま投げ直します(呼び出し側で「この件は諦めてスキップ/ログ」と判断できるように)。無限に粘らず**上限で打ち切る**のが安全設計です。

### 書いてみる

**課題**: 欠損に強い `safe_int` を使って、下の `stock_values`(在庫数の生文字列リスト。空文字が混じる)を**すべて整数のリスト**に変換して `result2` に入れてください。空文字は `0` にフォールバックします(期待値: `[12, 0, 7]`)。

ヒント(概念レベル): worked example で定義済みの `safe_int` をリスト内包表記で各要素に適用するだけ。`[safe_int(s) for s in stock_values]` の形。

In [ ]:
stock_values = ["12", "", "7"]

result2 = None
# ここに書く(result2 に代入する。safe_int を各要素に適用して整数のリストにする)


check("概念2: 欠損に強い変換", result2, [12, 0, 7],
      hint="[safe_int(s) for s in stock_values] の形。空文字は safe_int が 0 に変えてくれる")

---
## 概念3: CSV出力 — `csv.DictWriter` で「納品物」を作る

### なぜ学ぶか
スクレイピングの最終成果物は、たいてい**CSVファイル**です。Excelで開けて、他部署に渡せて、集計にかけられる — この「納品物」を作るのが [4 出力] 工程。ここで文字コードや改行を間違えると、Excelで文字化けしたり、1行が2行に割れたりします。標準ライブラリの `csv` モジュールに任せれば、ヘッダ生成・カンマの区切り・値の中にカンマが入ったときのクォート処理を全部やってくれます。

### 解説

`csv.DictWriter` は、C# の `CsvHelper` の `CsvWriter.WriteRecords(list)` に相当します。`list[dict]` を渡すと、`fieldnames`(列の順番)に従ってCSVを書き出してくれます。

```python
with open(path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "price"])
    writer.writeheader()          # 1行目: name,price
    writer.writerows(rows)        # 各 dict を1行ずつ
```

3つの約束事:
- **`encoding="utf-8"`**: 日本語を正しく保存(C# の `Encoding.UTF8`)。
- **`newline=""`**: これを付けないと Windows で改行が二重になり、1行おきに空行が入ります。csv を書くときの**必須のおまじない**。
- **`fieldnames`**: 列の順番と、どの列を出すかを固定します。`dict` のキーと一致させます。

`with open(...) as f:` は C# の `using (var f = ...)` と同じで、ブロックを抜けると自動でファイルを閉じます。

> **このレッスンでは実ファイルを散らかさないため `io.StringIO`(メモリ上の擬似ファイル)に書きます。** `StringIO` は「文字列を貯めるバッファをファイルのように扱える」もの。`open(path)` の代わりに `io.StringIO()` を渡せば、書き込み結果を `.getvalue()` で文字列として取り出せます。本番では `io.StringIO()` を `open("out.csv", ...)` に置き換えるだけです。

In [ ]:
# GOAL: list[dict] が、ヘッダ付きのCSVテキストに変換される様子を見る

# STEP 1: これが整形済みのデータ(概念1・2の成果)。1レコード = 1 dict
rows = [
    {"name": "エチオピア イルガチェフェ", "price": 1200, "stock": 12},
    {"name": "グアテマラ アンティグア", "price": 980,  "stock": 0},
]
fieldnames = ["name", "price", "stock"]   # 列の順番を固定

# STEP 2: 実ファイルの代わりにメモリ上のバッファ(StringIO)へ書く
buffer = io.StringIO()
writer = csv.DictWriter(buffer, fieldnames=fieldnames)
writer.writeheader()      # 1行目にヘッダ name,price,stock
writer.writerows(rows)    # 各 dict を1行ずつ

# STEP 3: 書けた中身を文字列として取り出して確認
csv_text = buffer.getvalue()
print("--- 生成されたCSV ---")
print(csv_text)
print("--- 行数(ヘッダ込み) ---")
print(len(csv_text.strip().splitlines()), "行")

### 予測してみよう

次のセルは、**名前にカンマが含まれる**レコード(`"モカ, ハラー"`)を1件だけCSVにします。CSVはカンマ区切りなので、値の中のカンマは列の区切りと衝突します。

**実行する前に予測**: `csv.DictWriter` はこのカンマをどう扱うでしょうか? 出力された行はどんな見た目になるでしょう(そのまま `モカ, ハラー`? それとも何かで囲まれる?)。

In [ ]:
# 予測してから実行!
buffer = io.StringIO()
writer = csv.DictWriter(buffer, fieldnames=["name", "price"])
writer.writeheader()
writer.writerow({"name": "モカ, ハラー", "price": 500})   # 値にカンマが入っている
print(buffer.getvalue())

値の中にカンマがあると `csv` モジュールが自動で `"..."` のダブルクォートで囲み、区切りと衝突しないようにしてくれます。これを手書きの `",".join(...)` でやろうとするとこの手のバグを踏むので、**必ず `csv` モジュールに任せる**のが鉄則です。

### 書いてみる

**課題**: 下の `rows` と `fieldnames` を使って、`io.StringIO()` に `csv.DictWriter` でヘッダ+各行を書き出し、**生成されたCSVテキスト**(`.getvalue()` の結果)を `result3` に入れてください。

期待値(この文字列とちょうど一致):
```
name,price
アイス,300
ホット,350
```
(各行末に改行があり、末尾にも改行が付きます)

ヒント(概念レベル): worked example の STEP 2〜3 と同じ流れ。`io.StringIO()` を作り → `csv.DictWriter(buffer, fieldnames=fieldnames)` → `writeheader()` → `writerows(rows)` → `result3 = buffer.getvalue()`。

In [ ]:
rows = [
    {"name": "アイス", "price": 300},
    {"name": "ホット", "price": 350},
]
fieldnames = ["name", "price"]

result3 = None
# ここに書く(result3 に代入する。StringIO に DictWriter で書き、getvalue() を入れる)


check("概念3: CSVを書き出す", result3, "name,price\r\nアイス,300\r\nホット,350\r\n",
      hint="io.StringIO() -> csv.DictWriter(buffer, fieldnames=fieldnames) -> writeheader() -> writerows(rows) -> buffer.getvalue()。csv は行末に \\r\\n を付ける")

---
## つなげてみる: 概念1→2→3 の一気通貫(次ユニットの予告編)

ここまでの3つ — クレンジング・欠損処理・CSV出力 — は、実際には**1本のパイプライン**として連なります。下は `DIRTY_HTML` から3レコードを取り出し、整形して、CSVにするミニ版です(演習 `ex04_capstone` で自分でこの全体を組みます)。読んで流れを掴んでください。

In [ ]:
# GOAL: 汚れHTML -> 解析 -> クレンジング -> CSV の全工程が1つに繋がるのを俯瞰する

# STEP 1: 各 <span class="xxx">中身</span> を取り出す小道具(素朴な正規表現)
def extract_span(block, cls):
    m = re.search(rf'<span class="{cls}">(.*?)</span>', block, re.DOTALL)
    return m.group(1) if m else ""

def safe_int(text, default=0):
    try:
        return int(text)
    except (TypeError, ValueError):
        return default

# STEP 2: <li class="record"> ブロックごとに、概念1のクレンジングをかけて dict にする
blocks = re.findall(r'<li class="record">(.*?)</li>', DIRTY_HTML, re.DOTALL)
records = []
for b in blocks:
    name = re.sub(r"[ 　]+", " ", extract_span(b, "name").strip(" 　"))
    price = int(extract_span(b, "price").replace("￥", "").replace("¥", "").replace("円", "").replace(",", "").strip())
    stock = safe_int(extract_span(b, "stock").strip())   # 空セルは 0 に
    records.append({"name": name, "price": price, "stock": stock})

for r in records:
    print(r)

# STEP 3: 概念3で CSV に。ここまでの全工程がこの数行に凝縮されている
buffer = io.StringIO()
w = csv.DictWriter(buffer, fieldnames=["name", "price", "stock"])
w.writeheader()
w.writerows(records)
print("--- 最終CSV ---")
print(buffer.getvalue())

汚れた入力が、そのままExcelで開ける整った表になりました。**unit06(キャップストーン)では、これを複数ページ・ページネーション・robots.txt尊重まで含めた本物のスクレイパー1本にまとめます。** 今日の3つの部品が、そこで全部つながります。

---
## 振り返り(1〜2文でOK — このセルを編集して書き込んでください)

- **今日学んだことを自分の言葉で**:
- **難しかったこと(あれば)**:

(この記述はセッション終了時にチューターが学習ノートとスキルレベル判定に使います)

## まとめと次へ

| 概念 | 一言で | C#で言うと |
|------|--------|-----------|
| データクレンジング | `strip`/`replace`/`re.sub` で空白・通貨・日付を正規化。ゴミを入れればゴミが出る | DTOマッピング前の正規化 |
| 例外処理 | `try/except` でフォールバック値を返す `safe_int` を自作 | `int.TryParse` |
| リトライ | 指数バックオフ+ジッタで粘り、上限で打ち切る。`sleep_fn` を注入してテスト可能に | `Polly` の `WaitAndRetry` + DI |
| CSV出力 | `csv.DictWriter` で `list[dict]` を UTF-8・`newline=""` で書き出す | `CsvHelper.WriteRecords` |

**この先どこで使うか**:
- **unit06(キャップストーン)** で、取得([1] requests)→解析([2] BeautifulSoup)→整形([3] 今日のクレンジング)→出力([4] 今日のCSV)を**1本のパイプライン**に統合します。今日作った `safe_int` や `csv.DictWriter` の使い方が、そのまま最終成果物のコードになります。
- 例外処理とリトライは、複数ページを巡回するとき「1ページ取得に失敗しても全体を止めない」ために必須です。壊れたレコードは**スキップしてログ**、通信失敗は**リトライ** — この2つが「夜に壊れないスクレイパー」の条件です。

**次**: 演習 `ex01_clean_fields.py` へ。lesson を見ながらで OK。テストは
`python -m pytest courses/web-scraping/unit05-robust-and-csv/tests/test_ex01.py -q`